In [1]:
import pandas as pd
from sqlalchemy import create_engine, text

In [2]:
engine = create_engine(
    "postgresql+psycopg://postgres:root"
    "@localhost:5432/olist_ecommerce"
)

In [3]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT version()"))
    print(result.scalar())

PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35229, 64-bit


In [19]:
database = {
    'customers': r"C:\Users\Admin\OneDrive\Documents\Sem-2\Project's\Olist_SQL_Project\data\olist_customers_dataset.csv",
    'orders': r"C:\Users\Admin\OneDrive\Documents\Sem-2\Project's\Olist_SQL_Project\data\olist_orders_dataset.csv",
    'order_items': r"C:\Users\Admin\OneDrive\Documents\Sem-2\Project's\Olist_SQL_Project\data\olist_order_items_dataset.csv",
    'payments': r"C:\Users\Admin\OneDrive\Documents\Sem-2\Project's\Olist_SQL_Project\data\olist_order_payments_dataset.csv",
    'products': r"C:\Users\Admin\OneDrive\Documents\Sem-2\Project's\Olist_SQL_Project\data\olist_products_dataset.csv",
    "sellers": r"C:\Users\Admin\OneDrive\Documents\Sem-2\Project's\Olist_SQL_Project\data\olist_sellers_dataset.csv",
    "geolocation": r"C:\Users\Admin\OneDrive\Documents\Sem-2\Project's\Olist_SQL_Project\data\olist_geolocation_dataset.csv",
    "reviews": r"C:\Users\Admin\OneDrive\Documents\Sem-2\Project's\Olist_SQL_Project\data\olist_order_reviews_dataset.csv",
    "product_category": r"C:\Users\Admin\OneDrive\Documents\Sem-2\Project's\Olist_SQL_Project\data\product_category_name_translation.csv"
}

for table_name, file_path in database.items():
    globals()[table_name] = pd.read_csv(file_path)
    print({table_name})
    globals()[table_name].info()
    print()

{'customers'}
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 11.0 MB

{'orders'}
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_app

In [20]:
customers["customer_zip_code_prefix"] = customers["customer_zip_code_prefix"].astype(str)
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
orders["order_approved_at"] = pd.to_datetime(orders["order_approved_at"])
orders["order_delivered_carrier_date"] = pd.to_datetime(orders["order_delivered_carrier_date"])
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"])
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"])
order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])
order_items["order_item_id"] = order_items["order_item_id"].astype(str)
sellers["seller_zip_code_prefix"] = sellers["seller_zip_code_prefix"].astype(str)
geolocation["geolocation_zip_code_prefix"] = geolocation["geolocation_zip_code_prefix"].astype(str)
reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"])


In [33]:
reviews.shape

(99224, 7)

In [ ]:
geolocation = geolocation.drop_duplicates(subset=["geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"], keep="first")
geolocation = (geolocation.groupby("geolocation_zip_code_prefix", as_index=False)
    .agg({
        "geolocation_lat": "median",
        "geolocation_lng": "median"}))

In [27]:
geolocation.shape

(19015, 3)

In [30]:
for table_name in database:
    globals()[table_name].to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",
        index=False,
        chunksize=5000,
        method="multi"
    )
    print(f"{table_name}' uploaded to database successfully rows {len(globals()[table_name])}.")

customers' uploaded to database successfully rows 99441.
orders' uploaded to database successfully rows 99441.
order_items' uploaded to database successfully rows 112650.
payments' uploaded to database successfully rows 103886.
products' uploaded to database successfully rows 32951.
sellers' uploaded to database successfully rows 3095.
geolocation' uploaded to database successfully rows 19015.
reviews' uploaded to database successfully rows 99224.
product_category' uploaded to database successfully rows 71.


In [38]:
from sqlalchemy import text

primary_keys = {
    "customers": ["customer_id"],
    "sellers": ["seller_id"],
    "orders": ["order_id"],
    "products": ["product_id"],
    "order_items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"],
    "reviews": ["review_id", "order_id"],
    "geolocation": ["geolocation_zip_code_prefix"]
}

with engine.begin() as conn:
    for table, columns in primary_keys.items():
        column_sql = ", ".join(f'"{col}"' for col in columns)

        conn.execute(text(
            f'ALTER TABLE "{table}" ADD PRIMARY KEY ({column_sql})'
        ))

        print(f"Primary key added: {table} -> {columns}")

Primary key added: customers -> ['customer_id']
Primary key added: sellers -> ['seller_id']
Primary key added: orders -> ['order_id']
Primary key added: products -> ['product_id']
Primary key added: order_items -> ['order_id', 'order_item_id']
Primary key added: payments -> ['order_id', 'payment_sequential']
Primary key added: reviews -> ['review_id', 'order_id']
Primary key added: geolocation -> ['geolocation_zip_code_prefix']


In [43]:
foreign_keys = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("order_items", "seller_id", "sellers", "seller_id"),
    ("payments", "order_id", "orders", "order_id"),
    ("reviews", "order_id", "orders", "order_id")
]

with engine.begin() as conn:
    for child, child_col, parent, parent_col in foreign_keys:
        constraint_name = f"fk_{child}_{parent}"

        conn.execute(text(
            f'ALTER TABLE "{child}" ADD FOREIGN KEY ("{child_col}") REFERENCES "{parent}" ("{parent_col}")'
        ))

        print(f"Foreign key added: {child}.{child_col} -> {parent}.{parent_col}")

Foreign key added: orders.customer_id -> customers.customer_id
Foreign key added: order_items.order_id -> orders.order_id
Foreign key added: order_items.product_id -> products.product_id
Foreign key added: order_items.seller_id -> sellers.seller_id
Foreign key added: payments.order_id -> orders.order_id
Foreign key added: reviews.order_id -> orders.order_id


In [45]:
def q(sql):
    return display(pd.read_sql_query(sql, engine.connect()))

In [46]:
q("SELECT * FROM customers LIMIT 5;")

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
